In [ ]:
# CELL 1 — IMPORTS
from sklearn.preprocessing import StandardScaler
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,        
    classification_report, 
    confusion_matrix,      
    roc_auc_score,        
    roc_curve              
)
import joblib  
import warnings
warnings.filterwarnings('ignore')
print('All libraries imported successfully!')

In [ ]:
# CELL 2 — LOAD & CLEAN DATA
df = pd.read_csv('heart_cleveland_upload.csv')

# Remove duplicate header row if present
df = df[df['age'] != 'age'].reset_index(drop=True)

# Convert all columns to numeric
df = df.apply(pd.to_numeric, errors='coerce')

print(df.head(2))
print(df.dtypes)

In [ ]:
# CELL 3 — EDA
print('basic info about dataset')
print(df.info())
print('statistical summary')
print(df.describe())
print('check the missing value in dataset')
print(df.isnull().sum())
print('check the target distribution')
print(df['condition'].value_counts())

In [ ]:
# CELL 4 — PREPROCESSING & HEATMAP

# Convert key columns to numeric
df['ca'] = pd.to_numeric(df['ca'], errors='coerce')
df['thal'] = pd.to_numeric(df['thal'], errors='coerce')
df['condition'] = pd.to_numeric(df['condition'], errors='coerce')

# Fill missing values with median
df['ca'] = df['ca'].fillna(df['ca'].median())
df['thal'] = df['thal'].fillna(df['thal'].median())

# Binarize target: 0 = no disease, 1 = disease
df['condition'] = df['condition'].apply(lambda x: 1 if x > 0 else 0)
print('after binarizing the data')
print(df['condition'].value_counts())

# Correlation heatmap
correlation_matrix = df.select_dtypes(include=['number']).corr()
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# CELL 5 — FEATURE MATRIX & SCALING
X = df.drop(columns=['condition'])
y = df['condition']

print(f'Feature matrix shape: {X.shape}')   # should be (298, 13)
print(f'Target vector shape:  {y.shape}')   # should be (298,)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print('Feature scaling done successfully!')
print(X_scaled.head())
print(X.dtypes)
print(X.head(2))

In [ ]:
# CELL 6 — TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set size: {X_train.shape[0]} samples')
print(f'Test set size:     {X_test.shape[0]} samples')

In [ ]:
# CELL 7 — EVALUATE MODEL HELPER FUNCTION
def evaluate_model(model, model_name, X_test, y_test):
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    print(f'\n{"="*50}')
    print(f' {model_name}')
    print(f'{"="*50}')
    print(f'  Accuracy : {acc:.4f}')
    print(f'  AUC-ROC  : {auc:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Disease', 'Disease'],
                yticklabels=['No Disease', 'Disease'])
    plt.title(f'Confusion Matrix — {model_name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{model_name.replace(" ", "_")}.png', dpi=150)
    plt.show()

    return acc, auc, y_pred

print('evaluate_model function defined!')

In [ ]:
# CELL 8 — TRAIN & COMPARE 3 MODELS

# Model 1: Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
lr_acc, lr_auc, _ = evaluate_model(lr_model, 'Logistic Regression', X_test, y_test)

# Model 2: Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_acc, rf_auc, _ = evaluate_model(rf_model, 'Random Forest', X_test, y_test)

# Model 3: Gradient Boosting
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)
gb_acc, gb_auc, _ = evaluate_model(gb_model, 'Gradient Boosting', X_test, y_test)

In [ ]:
# CELL 9 — ROC CURVE COMPARISON
plt.figure(figsize=(10, 7))

for model, name in [(lr_model, 'Logistic Regression'),
                    (rf_model, 'Random Forest'),
                    (gb_model, 'Gradient Boosting')]:
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_score   = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_score:.4f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curve_comparison.png', dpi=150)
plt.show()

In [ ]:
# CELL 10 — FEATURE IMPORTANCE (Random Forest)
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importance.values, y=feature_importance.index, palette='viridis')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

print('\nTop 5 Most Important Features:')
print(feature_importance.head())

In [ ]:
# CELL 11 — HYPERPARAMETER TUNING (GridSearchCV)
param_grid = {
    'n_estimators':      [50, 100, 200],
    'max_depth':         [5, 10, 15, None],
    'min_samples_split': [2, 5, 10]
}

print('Running GridSearchCV... (this may take a moment)')

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f'\nBest Hyperparameters: {grid_search.best_params_}')
print(f'Best Cross-Val AUC:   {grid_search.best_score_:.4f}')

best_model = grid_search.best_estimator_
best_acc, best_auc, _ = evaluate_model(best_model, 'Tuned Random Forest', X_test, y_test)

In [ ]:
# CELL 12 — CROSS VALIDATION
cv_scores = cross_val_score(
    best_model, X_scaled, y, cv=10, scoring='roc_auc'
)

print('10-Fold Cross-Validation AUC Scores:')
print(cv_scores)
print(f'Mean AUC : {cv_scores.mean():.4f}')
print(f'Std Dev  : {cv_scores.std():.4f}')

In [ ]:
# CELL 13 — SAVE MODEL & SCALER
joblib.dump(best_model, 'heart_disease_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print('Model and scaler saved!')
print('  → heart_disease_model.pkl')
print('  → scaler.pkl')

In [ ]:
# CELL 14 — PREDICT FOR A NEW PATIENT
loaded_model  = joblib.load('heart_disease_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')

def predict_heart_disease(patient_data):
    patient_df     = pd.DataFrame([patient_data])
    patient_df     = patient_df[X.columns]
    patient_scaled = loaded_scaler.transform(patient_df)
    prediction     = loaded_model.predict(patient_scaled)[0]
    probability    = loaded_model.predict_proba(patient_scaled)[0][1]

    if probability < 0.3:
        risk_level = 'LOW RISK'
    elif probability < 0.6:
        risk_level = 'MODERATE RISK'
    else:
        risk_level = 'HIGH RISK'

    return {
        'prediction':  prediction,
        'probability': round(probability, 4),
        'risk_level':  risk_level,
        'message':     'Heart disease detected.' if prediction == 1 else 'No heart disease detected.'
    }

# Sample patient
sample_patient = {
    'age': 63, 'sex': 1, 'cp': 3, 'trestbps': 145,
    'chol': 233, 'fbs': 1, 'restecg': 0, 'thalach': 150,
    'exang': 0, 'oldpeak': 2.3, 'slope': 0, 'ca': 0, 'thal': 1
}

result = predict_heart_disease(sample_patient)
print('\n' + '='*50)
print('PATIENT RISK ASSESSMENT')
print('='*50)
for key, value in result.items():
    print(f'  {key:12s}: {value}')

In [ ]:
# CELL 15 — FINAL SUMMARY
print('\n' + '='*60)
print('FINAL MODEL COMPARISON SUMMARY')
print('='*60)
print(f'{"Model":<28} {"Accuracy":>10} {"AUC-ROC":>10}')
print('-'*50)
print(f'{"Logistic Regression":<28} {lr_acc:>10.4f} {lr_auc:>10.4f}')
print(f'{"Random Forest":<28} {rf_acc:>10.4f} {rf_auc:>10.4f}')
print(f'{"Gradient Boosting":<28} {gb_acc:>10.4f} {gb_auc:>10.4f}')
print(f'{"Tuned Random Forest":<28} {best_acc:>10.4f} {best_auc:>10.4f}')
print('='*60)
print('\nAll charts saved as PNG files.')
print("Best model saved as 'heart_disease_model.pkl'")